# 03 - Data Cleaning


In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
pd.set_option('display.max_columns', 50)

from src.data.load_data import load_csv
from src.data.data_quality import data_quality_summary
from src.cleaning.missing_values import missing_value_summary
from src.cleaning.duplicates import count_duplicates
from src.cleaning.outliers import iqr_outlier_report
from src.cleaning.clean_data import clean_dataset
from src.utils.config import load_config, resolve_path
from src.utils.helpers import save_dataframe

config = load_config()
df_raw = load_csv(resolve_path(config['data']['raw_path']))
print(f"Raw dataset shape: {df_raw.shape}")

2026-09-21 00:06:01 | INFO     | src.data.load_data | Loaded CSV 'manufacturing_defect_dataset.csv' with shape (3240, 17)


Raw dataset shape: (3240, 17)


## Step 1: Missing values


In [2]:
summary = missing_value_summary(df_raw)
summary

{'dataframe':       ProductionVolume  ProductionCost  SupplierQuality  DeliveryDelay  \
 0                  202    13175.403783        86.648534              1   
 1                  535    19770.046093        86.310664              4   
 2                  960    19060.820997        82.132472              0   
 3                  370     5647.606037        87.335966              5   
 4                  206     7472.222236        81.989893              3   
 ...                ...             ...              ...            ...   
 3235               762    11325.689263        89.252385              2   
 3236               335     5598.837988        95.701437              4   
 3237               835    11736.177712        96.431554              5   
 3238               302    13664.196210        91.089782              1   
 3239               355    13563.605806        83.595956              2   
 
       DefectRate  QualityScore  MaintenanceHours  DowntimePercentage  \
 0       3.1

**Cleaning decision:** No missing values were found, so no imputation or row removal is necessary for this dataset.

## Step 2: Duplicate rows


In [3]:
dup_count = count_duplicates(df_raw)
print(f"Duplicate rows found: {dup_count}")

Duplicate rows found: 0


**Cleaning decision:** No duplicate rows were found, so no rows need to be removed.

## Step 3: Data type validation


In [4]:
non_numeric = [c for c in df_raw.columns if not pd.api.types.is_numeric_dtype(df_raw[c])]
print(f"Non-numeric columns: {non_numeric if non_numeric else 'None - all columns are numeric.'}")

Non-numeric columns: None - all columns are numeric.


**Cleaning decision:** All 17 columns are already numeric; no type conversion is required.

## Step 4: Outlier investigation (IQR method)



In [5]:
outlier_report = iqr_outlier_report(df_raw, multiplier=config['analysis']['iqr_multiplier'])
outlier_report

2026-09-21 00:06:01 | INFO     | src.cleaning.outliers | Outlier report generated for 17 numeric column(s).


,outlier_count,outlier_percentage,lower_bound,upper_bound
column,,,,
DefectStatus,517,15.96,1.0000,1.0000
ProductionVolume,0,0.00,-357.8750,1455.1250
ProductionCost,0,0.00,-2364.6204,27217.9121
DeliveryDelay,0,0.00,-3.5000,8.5000
SupplierQuality,0,0.00,69.9881,109.6710
QualityScore,0,0.00,39.7278,120.7294
MaintenanceHours,0,0.00,-11.1250,33.8750
DowntimePercentage,0,0.00,-2.5008,7.5403
DefectRate,0,0.00,-1.8617,7.3643


**Cleaning decision on outliers:** Every numeric column in this dataset represents a genuine operational measurement (a cost, a duration, a percentage, a count) with a plausible business range. Values flagged as statistical outliers here - for example unusually high `MaintenanceHours` or `SafetyIncidents` - are far more likely to represent real, informative manufacturing events (a long maintenance intervention, an unusually problematic batch) than a data entry mistake. Removing them would risk deleting exactly the kind of signal a defect-driver analysis is trying to find.



## Step 5: Applying the full cleaning pipeline



In [6]:
df_clean = clean_dataset(df_raw)
print(f"Shape before cleaning: {df_raw.shape}")
print(f"Shape after cleaning:  {df_clean.shape}")

2026-09-21 00:06:01 | INFO     | src.cleaning.clean_data | Starting data cleaning process...
2026-09-21 00:06:01 | INFO     | src.cleaning.duplicates | Removed 0 duplicate row(s) (keep='first').
2026-09-21 00:06:01 | INFO     | src.cleaning.clean_data | Cleaning complete. Missing-value decision: No missing values were found. No missing-value handling was necessary. | Duplicate rows removed: 0


Shape before cleaning: (3240, 17)
Shape after cleaning:  (3240, 17)


**Interpretation:** As expected, the shape is unchanged - this cleaning run confirms, rather than changes, the dataset's integrity.

## Step 6: Saving the interim (cleaned) dataset


In [7]:
interim_path = resolve_path(config['data']['interim_path'])
save_dataframe(df_clean, interim_path)
print(f"Interim cleaned dataset saved to: {interim_path}")

Interim cleaned dataset saved to: E:\Python projects\manufacturing-quality-operational-performance-analysis\data\interim\manufacturing_interim.csv


## Summary of cleaning decisions

| Issue | Found? | Action Taken |
|---|---|---|
| Missing values | No (0 across all columns) | None required |
| Duplicate rows | No (0 duplicate rows) | None required |
| Non-numeric columns | No (all 17 columns numeric) | None required |
| Invalid (negative) values | No | None required |
| Statistical outliers (IQR) | Some columns show IQR outliers | **Kept, not removed** - considered genuine operational variation, not data errors |
